# MechVerify V4 — Qwen + Python Auditor (No Google Drive)

This notebook **never mounts or accesses Google Drive**. You do not need a Qwen installation on your own computer or an LLM API key.

1. In Colab, choose **Runtime → Change runtime type → T4 GPU** (if available).
2. Run the cells top-to-bottom.
3. Download the resulting JSON file to your computer at the end.

**Important:** Colab's `/content` directory is temporary. Files can disappear when the runtime resets, so download results before leaving. If the browser asks for permission to download a file, that is separate from Google Drive access.

## 1. Check GPU

In [ ]:
import torch
print('CUDA GPU available:', torch.cuda.is_available())
print('Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU (model may run slowly)')

## 2. Get the latest GitHub code and install Qwen dependencies

If the project directory already exists in this Colab runtime, we update it instead of cloning twice.

In [ ]:
from pathlib import Path
import subprocess
repo_dir = Path('/content/MechVerify-COF')
if repo_dir.exists():
    subprocess.run(['git', '-C', str(repo_dir), 'pull', '--ff-only'], check=True)
else:
    subprocess.run(['git', 'clone', 'https://github.com/bao-yutian/MechVerify-COF.git', str(repo_dir)], check=True)
%cd /content/MechVerify-COF
%pip install -q -U transformers accelerate

## 3. Run CPU-only unit tests

These tests check program behavior. Passing them does not establish the Auditor's scientific accuracy.

In [ ]:
!python -m unittest discover -s tests -v

## 4. Run a three-case Qwen smoke test

The first model download is approximately 3 GB. `-u` streams progress and each completed case is saved immediately to `results/v4_smoke.json`. This is a **new V4** extraction + evidence check, not the old Qwen-only classification experiment.

In [ ]:
!python -u -m mechverify.v4_qwen --limit 3 --output results/v4_smoke.json

### Inspect and download the 3-case result

Download the file even if you decide not to run all 15 cases.

In [ ]:
import json
from pathlib import Path
from google.colab import files
smoke_path = Path('results/v4_smoke.json')
if smoke_path.exists():
    smoke_result = json.loads(smoke_path.read_text(encoding='utf-8'))
    print(json.dumps(smoke_result.get('summary', {}), indent=2, ensure_ascii=False))
    files.download(str(smoke_path))
else:
    print('Run the smoke test first; no output file exists yet.')

## 5. Run all 15 COF claims

Run this **only after** the smoke test works. Results are saved after every case in Colab's temporary `results/v4_all_15.json`. Model loading may take time again.

In [ ]:
!python -u -m mechverify.v4_qwen --output results/v4_all_15.json

## 6. Inspect & download the full report

The report contains the raw Qwen response, extracted structure, final evidence-checker decision and provisional gold label for each case.

In [ ]:
import json
from pathlib import Path
from google.colab import files
report_path = Path('results/v4_all_15.json')
if report_path.exists():
    report = json.loads(report_path.read_text(encoding='utf-8'))
    print('Report status:', report.get('status', 'unknown'))
    print(json.dumps(report.get('summary', {}), indent=2, ensure_ascii=False))
    for result in report.get('predictions', []):
        if result.get('predicted') != result.get('gold_label'):
            print(result['id'], ':', result['predicted'], '(provisional gold:', result['gold_label'] + ')')
    files.download(str(report_path))
else:
    print('No full report yet. Run Step 5, or download the smoke test report.')

## Optional: replay saved extractions without loading Qwen

If you upload a previously saved **V4** JSON report to Colab later, this evaluates its existing extractions with the current Python checker. It does not call Qwen. The old V3 or Qwen-only result file cannot be replayed with this command.

In [ ]:
# Optional — execute after uploading v4_all_15.json again if needed:
# !python -m mechverify.v4_qwen --replay results/v4_all_15.json --output results/v4_replay.json